# Food Suggest MLOps smoke training
Runs the production DVC pipeline for three epochs with an isolated W&B artifact.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, subprocess

secrets = UserSecretsClient()
required = [
    'DVC_S3_REMOTE_URL', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY',
    'AWS_DEFAULT_REGION', 'WANDB_API_KEY', 'WANDB_ENTITY'
]
for name in required:
    os.environ[name] = secrets.get_secret(name)
os.environ['WANDB_PROJECT'] = 'ingredient-detection'
os.environ['WANDB_RUN_NAME'] = 'mlops-smoke-3ep'
os.environ['WANDB_MODEL_ARTIFACT'] = 'ingredient-detector-smoke'
os.environ['EXECUTION_ENV'] = 'kaggle-smoke'

repo_dir = '/kaggle/working/cook-smart'
if os.path.isdir(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/HTM0410/cook-smart.git', repo_dir], check=True)
os.chdir(repo_dir)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-mlops.txt'], check=True)


In [ ]:
subprocess.run(['python', '-m', 'mlops.configure_dvc_s3'], check=True)
from pathlib import Path
dataset_targets = sorted(str(path) for path in Path('mlops/data/yolo_dataset').glob('*.dvc'))
if not dataset_targets:
    raise FileNotFoundError('No dataset .dvc tracking file found')
subprocess.run(['dvc', 'pull', *dataset_targets], check=True)
subprocess.run([
    'dvc', 'exp', 'run',
    '--set-param', 'train.epochs=3',
    '--set-param', 'train.patience=2'
], check=True)
subprocess.run(['dvc', 'metrics', 'show'], check=True)
